# High-res SAM-Road (UNet + RGB) on Kaggle

Trains `unet_rgb_highres` on the 2.5m `high_res_rgb/` aerial tiles: 512x512
patches, imagery read at native resolution (no upsampling), labels
re-rasterised at 2.5m, and road/keypoint labels masked out wherever the imagery
is zero-padded near the edge of the country.

**Before running**, attach `kelvinwei/rosadatasethighres` under *Add Input*.
That one dataset (20.1 GB) is the complete ROSA set — it carries `splits/` and
all six per-split folders including `high_res_rgb/`, nested one level under
`ROSADataset/`, so nothing else needs attaching.

The prep cell resolves that nesting itself. If you ever split the high-res tiles
into their own dataset, attach both and it will stitch them together; either way
it prints what it found and what is missing before you commit to a long session.

**Sessions are capped at ~12h and 100 epochs will not fit in one.** Training is
time-boxed and writes `last.ckpt` every epoch; the *Resume* section explains how
to carry on in the next session.

## Clone the repo

In [ ]:
REPO_USER = "kelviy"
REPO_NAME = "GeoSAMRoad"
BRANCH    = "highres"          # branch carrying the high-res dataset pipeline

# public repo, so no PAT needed
%cd /kaggle/working
!rm -rf /kaggle/working/{REPO_NAME}
!git clone --recursive -b {BRANCH} https://github.com/{REPO_USER}/{REPO_NAME}.git
%cd /kaggle/working/{REPO_NAME}
!git log --oneline -1
%cd /kaggle/working

## Dependencies + dataset

Installs the `kaggle` extra, pinned to the host image's own
**torch/torchvision/numpy**. Both pins matter: replacing torch silently drops
CUDA, and letting numpy float breaks the image's prebuilt `numba` so pip tries
to rebuild it from source and the install dies.

The `kaggle` extra also leaves out `terratorch` — it is only needed for the
TerraMind backbone, and its dependency tree is what pushes numpy up. UNet never
imports it. Pass `EXTRA=geosamroad` if you do want TerraMind here.

Then resolves `DATASET_DIR`: it walks every mount under `/kaggle/input`, works
out which one supplies each piece, and stitches them into a single root of
symlinks at `/kaggle/working/dataset`.

Takes several minutes.

In [ ]:
!bash /kaggle/working/GeoSAMRoad/src/geosamroad/scripts/kaggle/prep_env.bash

### Check the dataset resolved

Cheap sanity check before committing to a long run. Builds one real training
sample end to end, which is where a missing `mask_adj_graphs/` or a truncated
upload would otherwise surface — several minutes into the first epoch.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, "/kaggle/working/GeoSAMRoad/src")
DATASET_DIR = Path("/kaggle/working/geosamroad_dataset_dir.txt").read_text().strip()
print("DATASET_DIR:", DATASET_DIR)

for split in ("train", "val", "test"):
    df = pd.read_csv(Path(DATASET_DIR) / "splits" / f"{split}.csv")
    print(f"  {split:5s} {len(df):5d} tiles")

needed = {"image_path", "mask_path", "mask_graph_path", "mask_adj_graph_path"}
missing = needed - set(df.columns)
print("sidecar columns:", "all present" if not missing else f"MISSING {sorted(missing)}")

from geosamroad.utils import load_config, finalize_config, build_dataset_config
from geosamroad.dataset.samroad_dataset import SAMROAD_Dataset

cfg = load_config("/kaggle/working/GeoSAMRoad/src/geosamroad/configs/kaggle/unet_rgb_highres.yaml")
cfg.DATASET_DIR = DATASET_DIR
cfg.PRELOAD_GRAPHS = False
finalize_config(cfg)

ds = SAMROAD_Dataset(build_dataset_config(cfg), "train")
sample = ds[0]
print(f"\ntiles after dropping any without a high-res image: {len(ds.df)}")
print("patch size:", cfg.PATCH_SIZE, "| sample shapes:")
for k, v in sample.items():
    print(f"   {k:14s} {tuple(v.shape)} {v.dtype}")
assert tuple(sample["image"].shape) == (3, 512, 512)
assert sample["image"].max() > 1.5, "expected 0-255 uint8-valued RGB"
print("\nOK")

### Look at one sample

Worth 20 seconds: if the mask were misaligned with the imagery the run would
still train happily and just produce a bad model.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

rgb = sample["image"].numpy().transpose(1, 2, 0).astype(np.uint8)
road = sample["road_mask"].numpy() > 0
kp = sample["keypoint_mask"].numpy() > 0
over = rgb.copy()
over[road] = (0.45 * over[road] + 0.55 * np.array([255, 60, 60])).astype(np.uint8)
over[kp] = (0.2 * over[kp] + 0.8 * np.array([60, 255, 60])).astype(np.uint8)

fig, ax = plt.subplots(1, 2, figsize=(13, 6.6))
ax[0].imshow(rgb); ax[0].set_title("high_res_rgb 512x512 @2.5m")
ax[1].imshow(over); ax[1].set_title("road (red) + keypoints (green)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## Weights & Biases

Skip this cell to train without logging — the config also accepts
`--set WANDB_MODE=disabled`.

In [ ]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

wandb_api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login(key=wandb_api_key)

## Resume

`/kaggle/working` is wiped between sessions. To continue a run:

1. **Save Version** on the session you want to continue (its output is kept).
2. In the next session, *Add Input* → *Notebook Output* → that version.
3. Run this cell — it copies the newest `last.ckpt` it finds in
   `/kaggle/input/**/checkpoints` into `/kaggle/working/checkpoints`.

On a first run it finds nothing and training starts from scratch, which is fine.

In [ ]:
import shutil
from pathlib import Path

CKPT_DIR = Path("/kaggle/working/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

found = sorted(Path("/kaggle/input").glob("**/checkpoints/last.ckpt"),
               key=lambda p: p.stat().st_mtime, reverse=True)
RESUME = ""
if (CKPT_DIR / "last.ckpt").exists():
    RESUME = str(CKPT_DIR / "last.ckpt")
    print("resuming from this session's checkpoint:", RESUME)
elif found:
    for src in found[0].parent.glob("*.ckpt"):
        shutil.copy2(src, CKPT_DIR / src.name)
        print("copied", src.name)
    RESUME = str(CKPT_DIR / "last.ckpt")
    print("resuming from:", found[0])
else:
    print("no previous checkpoint found -- training from scratch")

## Train

`MAX_TRAIN_HOURS` stops the fit at the first epoch boundary past that budget, so
validation runs and `last.ckpt` is complete before the session is killed. The
budget is per session — it is **not** restored from the checkpoint — so every
resumed session gets the full allowance again.

Leave `TRAIN_EPOCHS=100` in every session: the epoch counter is restored from the
checkpoint, so each session picks up where the last one stopped and the run ends
when epoch 100 is reached, however many sessions that takes.

Drop `BATCH_SIZE` to 8 (with `ACCUMULATE_GRAD_BATCHES=2`) if the session hands
you a card smaller than 16GB.

In [ ]:
import subprocess, sys

DATASET_DIR = open("/kaggle/working/geosamroad_dataset_dir.txt").read().strip()
RESUME = globals().get("RESUME", "")   # empty unless the Resume cell was run

cmd = [
    sys.executable, "-m", "geosamroad.train",
    "--config", "src/geosamroad/configs/kaggle/unet_rgb_highres.yaml",
    "--accelerator", "gpu", "--devices", "1",
    "--dataset-dir", DATASET_DIR,
    "--set", "TRAIN_EPOCHS=100",
    "--set", "BATCH_SIZE=16",
    "--set", "ACCUMULATE_GRAD_BATCHES=1",
    "--set", "DATA_WORKER_NUM=2",
    "--set", "MAX_TRAIN_HOURS=10.5",         # per session, inside the ~12h cap
    "--set", "SEED=42",
]
if RESUME:
    cmd += ["--resume", RESUME]

print(" ".join(cmd), "\n")
subprocess.run(cmd, cwd="/kaggle/working/GeoSAMRoad", check=True)

## Threshold sweep on val

Sweeps the keypoint / road / topo probability thresholds on **val** and writes
them to `/kaggle/working/val_thresholds.yaml`. Run this only once training has
actually finished its 100 epochs — thresholds from a part-trained model are not
worth carrying forward.

In [ ]:
import subprocess, sys, glob, os

DATASET_DIR = open("/kaggle/working/geosamroad_dataset_dir.txt").read().strip()
best = sorted(glob.glob("/kaggle/working/checkpoints/epoch*.ckpt"),
              key=os.path.getmtime, reverse=True)
assert best, "no epoch*.ckpt -- train first"
print("checkpoint:", best[0])

subprocess.run([
    sys.executable, "-m", "geosamroad.test",
    "--config", "src/geosamroad/configs/kaggle/unet_rgb_highres.yaml",
    "--checkpoint", best[0], "--split", "val",
    "--accelerator", "gpu", "--devices", "1",
    "--dataset-dir", DATASET_DIR,
    "--set", "PRETRAINED=false",
    "--set", "THRESHOLD_SWEEP_OUT=/kaggle/working/val_thresholds.yaml",
], cwd="/kaggle/working/GeoSAMRoad", check=True)

print(open("/kaggle/working/val_thresholds.yaml").read())

## Inference on test

Writes predicted graphs to `/kaggle/working/infer_test/graph/*.p` plus fused
masks and visualisations. The graph metrics (APLS/TOPO) are deliberately not run
here — take these `.p` files to the HPC for that.

In [ ]:
import subprocess, sys, yaml, glob, os

DATASET_DIR = open("/kaggle/working/geosamroad_dataset_dir.txt").read().strip()
thr = yaml.safe_load(open("/kaggle/working/val_thresholds.yaml"))
best = sorted(glob.glob("/kaggle/working/checkpoints/epoch*.ckpt"),
              key=os.path.getmtime, reverse=True)[0]

subprocess.run([
    sys.executable, "-m", "geosamroad.inferencer",
    "--config", "src/geosamroad/configs/kaggle/unet_rgb_highres.yaml",
    "--checkpoint", best, "--split", "test",
    "--output-dir", "/kaggle/working/infer_test", "--device", "cuda",
    "--dataset-dir", DATASET_DIR,
    "--set", f"ITSC_THRESHOLD={thr['ITSC_THRESHOLD']}",
    "--set", f"ROAD_THRESHOLD={thr['ROAD_THRESHOLD']}",
    "--set", f"TOPO_THRESHOLD={thr['TOPO_THRESHOLD']}",
], cwd="/kaggle/working/GeoSAMRoad", check=True)